# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to explore and process a FAIR^2 dataset published using the [Croissant](https://mlcommons.org/croissant/) schema via the [`mlcroissant`](https://pypi.org/project/mlcroissant/) Python library.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# If not already installed, install the mlcroissant library.
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and initialize the Croissant package using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant metadata and Dataset object
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
# Display dataset name and description
print(f"Dataset Title: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview

Let's inspect available record sets, fields, and their Croissant `@id`s. These IDs will be used to reference all entities (record sets, fields/columns) in later operations.

In [ ]:
# List available record sets using their @id
record_sets = dataset.list_record_sets()  # returns a list of record set @ids
if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    print("Record sets available (by @id):")
    for rs_id in record_sets:
        print(f"  • {rs_id}")
    # For demonstration, show fields/columns for the first record set
    record_set_id = record_sets[0]
    print(f"\nFields/columns for record set '{record_set_id}':")
    fields = dataset.list_fields(record_set_id)  # returns a list of field/column @ids
    for field_id in fields:
        print(f"  ◦ {field_id}")
        desc = dataset.get_field_description(record_set=record_set_id, field=field_id)
        print(f"      ↳ {desc}")

## 3. Data Extraction

Load data from selected record sets into Pandas DataFrames. Reference **record sets** and **fields** only by their `@id`. Modify the variables as needed for your further analysis.

In [ ]:
# Choose record sets to extract (using their @id)
# For this notebook, extract data from all available record sets
record_sets_to_extract = dataset.list_record_sets()
dataframes = {}

for recset_id in record_sets_to_extract:
    recs = list(dataset.records(record_set=recset_id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[recset_id] = df
        print(f"→ Loaded {len(df)} records for record set {recset_id}")
        print("Columns (field @ids):", df.columns.tolist())
        display(df.head(3))
    else:
        print(f"No records extracted for record set {recset_id}")

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing operations such as filtering, normalization, and grouping, all using Croissant `@id` for field access.

> Edit the variables below to suit your use case if different variables are present in your dataset.

In [ ]:
# For demonstration, select the first non-empty record set
if dataframes:
    main_record_set_id = next(iter(dataframes))
    df = dataframes[main_record_set_id]
    print(f"Selected record set: {main_record_set_id}")
    
    # Identify numeric fields by checking dtype or infer from field description
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if not numeric_field_id:
        print("No numeric fields detected. Inspecting the dataframe:")
        display(df.head())
    else:
        print(f"Performing EDA on numeric field: {numeric_field_id}")
        # Remove outliers (example: filter > threshold)
        threshold = df[numeric_field_id].quantile(0.95)
        filtered_df = df[df[numeric_field_id] < threshold].copy()
        print(f"Filtered records where {numeric_field_id} < {threshold:.2f} (95th percentile): {len(filtered_df)} rows")

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If a grouping/categorical field exists, group by that field
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No non-numeric group field found to group data by.")
else:
    print("No dataframes loaded for analysis.")

## 5. Visualization

Visualize numeric and categorical relationships using the fields' Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals() and numeric_field_id:
    # Boxplot of the main numeric field
    plt.figure(figsize=(6,3))
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f'Boxplot of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # Histogram of the normalized numeric field
    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(6,3))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20)
        plt.title(f'Normalized {numeric_field_id} Distribution')
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.axvline(0, color='red', linestyle='--', lw=1)
        plt.show()

    # If group_field is present, barplot of mean values by group
    if 'group_field_id' in locals() and group_field_id:
        group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(
            data=group_means,
            x=group_field_id,
            y=numeric_field_id,
        )
        plt.xticks(rotation=60, ha='right')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("Unable to plot: missing numeric data or dataframes.")

## 6. Conclusion

- This notebook showed how to access and explore a FAIR^2 (Croissant) dataset using the `mlcroissant` Python library, referencing all entities by their `@id`.
- We examined metadata, listed record sets and fields, loaded records into DataFrames, and performed simple exploratory and visualization steps using the Croissant schema's identifiers.
- For detailed research analyses, consult the field and record set IDs and Croissant metadata for accurate variable interpretation and mapping.